# Importando as bibliotecas

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import mlflow
import optuna

import warnings
warnings.filterwarnings("ignore")

# Carregando a base de dados

In [ ]:
data = pd.read_csv('data/bike.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  object 
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  cnt         17379 non-null  int64  
dtypes: float64(4), int64(10), object(1)
memory usage: 2.0+ MB


In [3]:
data.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,1/1/2011,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,16
1,2,1/1/2011,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,40
2,3,1/1/2011,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,32
3,4,1/1/2011,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,13
4,5,1/1/2011,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,1


# Preparando o Dataset - Treino e Teste

In [4]:
# Converte a coluna 'dteday' para o tipo datetime
data['dteday'] = pd.to_datetime(data['dteday'])

In [5]:
# Divisão 80% treino e 20% teste
train = data.loc[data['dteday']< '2012-08-10']
test = data.loc[data['dteday']>= '2012-08-10']

In [6]:
# Porcentagem dos dados de treino
round(train.shape[0] / data.shape[0], 2)

0.8

In [7]:
train.drop(['instant', 'dteday'], axis=1, inplace=True)
test.drop(['instant', 'dteday'], axis=1, inplace=True)

# Validação Cruzada p/ teste de Modelos

In [8]:
# Separando as features e o target de treino
features_train = train.drop('cnt', axis=1)
target_train = train['cnt']

# Separando as features e o target de teste
features_test = test.drop('cnt', axis=1)
target_test = test['cnt']

In [24]:
def cross_validate_rmse(model, X, y, cv=5):
    scores = cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=cv)
    rmse_scores = -scores
    mean_rmse = rmse_scores.mean()
    std_rmse = rmse_scores.std()
    return rmse_scores, mean_rmse, std_rmse

# Decision Tree
rmse_scores, mean_rmse, std_rmse = cross_validate_rmse(DecisionTreeRegressor(random_state=42), features_train, target_train)
print(f'Decision Tree: RMSE médio: {mean_rmse:.2f}, Desvio padrão: {std_rmse:.2f}')

# Random Forest
rmse_scores, mean_rmse, std_rmse = cross_validate_rmse(RandomForestRegressor(n_estimators=1000, random_state=42, n_jobs=-1), features_train, target_train)
print(f'Random Forest: RMSE médio: {mean_rmse:.2f}, Desvio padrão: {std_rmse:.2f}')

# XGBoost with DataFrame
rmse_scores, mean_rmse, std_rmse = cross_validate_rmse(xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse', seed=42), features_train, target_train)
print(f'XGBoost (DataFrame): RMSE médio: {mean_rmse:.2f}, Desvio padrão: {std_rmse:.2f}')

# XGBoost with Dmatrix
dtrain = xgb.DMatrix(features_train, label=target_train)
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'seed': 42
}
cv_results = xgb.cv(params, dtrain, num_boost_round=1000, nfold=5, early_stopping_rounds=10)
mean_rmse = cv_results['test-rmse-mean'].min()
std_rmse = cv_results['test-rmse-std'].min()
print(f'XGBoost (DMatrix): RMSE médio: {mean_rmse:.2f}, Desvio padrão: {std_rmse:.2f}')


Decision Tree: RMSE médio: 90.62, Desvio padrão: 15.20
Random Forest: RMSE médio: 71.25, Desvio padrão: 14.70
XGBoost (DataFrame): RMSE médio: 62.06, Desvio padrão: 7.56
XGBoost (DMatrix): RMSE médio: 36.87, Desvio padrão: 1.39


# Experiment Tracking com MLflow e XGBoost

In [ ]:
# Instancia o MLflow
mlflow.set_experiment("bike_rental")
mlflow.set_tracking_uri("http://localhost:5000")

2025/02/19 21:37:03 INFO mlflow.tracking.fluent: Experiment with name 'bike_rental' does not exist. Creating a new experiment.


In [27]:
# Otimização de hiperparâmetros com Optuna e registro no MLflow

def objective(trial):
    # Define os hiperparâmetros a serem otimizados
    param = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': 42,
        'max_depth': trial.suggest_int("max_depth", 3, 10),
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.3),
        'gamma': trial.suggest_float("gamma", 0.0, 5.0),
        'min_child_weight': trial.suggest_int("min_child_weight", 1, 10),
        'subsample': trial.suggest_float("subsample", 0.5, 1.0),
        'colsample_bytree': trial.suggest_float("colsample_bytree", 0.5, 1.0)
    }
    
    # Inicia um run MLflow aninhado para registrar a trial corrente
    with mlflow.start_run(nested=True):
        mlflow.log_params(param)
        
        # Executa a validação cruzada com xgb.cv usando dtrain (definido em célula anterior)
        cv_results = xgb.cv(
            params=param,
            dtrain=dtrain,
            num_boost_round=1000,
            nfold=5,
            early_stopping_rounds=10,
            verbose_eval=False
        )
        
        # Retorna o menor RMSE obtido na validação cruzada
        best_rmse = cv_results['test-rmse-mean'].min()
        mlflow.log_metric("best_rmse", best_rmse)
    
    return best_rmse

# Cria um estudo Optuna e otimiza a função objetivo
study_dmatrix = optuna.create_study(direction="minimize")
study_dmatrix.optimize(objective, n_trials=50)

print("Melhor resultado:")
print(study_dmatrix.best_trial)

[I 2025-02-19 22:14:46,254] A new study created in memory with name: no-name-a5e86e06-9ef5-45d8-8268-661bf0586388
[I 2025-02-19 22:14:54,942] Trial 0 finished with value: 43.234122876565266 and parameters: {'max_depth': 3, 'learning_rate': 0.17927639741854914, 'gamma': 4.44202582950418, 'min_child_weight': 10, 'subsample': 0.5092200494944793, 'colsample_bytree': 0.9346162116685763}. Best is trial 0 with value: 43.234122876565266.


🏃 View run chill-cow-880 at: http://localhost:5000/#/experiments/626468295712000692/runs/7d08b51792034bb7a7e8da92cc9994fd
🧪 View experiment at: http://localhost:5000/#/experiments/626468295712000692


[I 2025-02-19 22:14:57,792] Trial 1 finished with value: 35.92841371107319 and parameters: {'max_depth': 10, 'learning_rate': 0.12951246155361593, 'gamma': 1.8402041944751861, 'min_child_weight': 1, 'subsample': 0.6093854409236971, 'colsample_bytree': 0.8547916840526999}. Best is trial 1 with value: 35.92841371107319.


🏃 View run spiffy-ram-555 at: http://localhost:5000/#/experiments/626468295712000692/runs/6a3ce8ca92fc4740bdd19fa60e32f0c8
🧪 View experiment at: http://localhost:5000/#/experiments/626468295712000692


[I 2025-02-19 22:14:59,459] Trial 2 finished with value: 36.35340254880178 and parameters: {'max_depth': 10, 'learning_rate': 0.1914364178108559, 'gamma': 1.7943784685637048, 'min_child_weight': 9, 'subsample': 0.6609401307082885, 'colsample_bytree': 0.8701249211865612}. Best is trial 1 with value: 35.92841371107319.


...

🏃 View run charming-stoat-854 at: http://localhost:5000/#/experiments/626468295712000692/runs/585bd7e127044808a40123af4cba571a
🧪 View experiment at: http://localhost:5000/#/experiments/626468295712000692


[I 2025-02-19 22:20:35,702] Trial 48 finished with value: 34.52624376393399 and parameters: {'max_depth': 9, 'learning_rate': 0.03981888185315734, 'gamma': 2.722685711732997, 'min_child_weight': 10, 'subsample': 0.6406041478967199, 'colsample_bytree': 0.9412702412739683}. Best is trial 35 with value: 34.34207656467384.


🏃 View run angry-seal-75 at: http://localhost:5000/#/experiments/626468295712000692/runs/441e151b954d4281b07c179768dfd4c3
🧪 View experiment at: http://localhost:5000/#/experiments/626468295712000692


[I 2025-02-19 22:20:39,045] Trial 49 finished with value: 35.08777231989015 and parameters: {'max_depth': 8, 'learning_rate': 0.09495771770587279, 'gamma': 2.774534188635514, 'min_child_weight': 8, 'subsample': 0.6374899455834234, 'colsample_bytree': 0.9354613993146264}. Best is trial 35 with value: 34.34207656467384.


🏃 View run carefree-mare-791 at: http://localhost:5000/#/experiments/626468295712000692/runs/59e5593a6ce3492b90398538deec156a
🧪 View experiment at: http://localhost:5000/#/experiments/626468295712000692
Melhor resultado:
FrozenTrial(number=35, state=TrialState.COMPLETE, values=[34.34207656467384], datetime_start=datetime.datetime(2025, 2, 19, 22, 19, 16, 580193), datetime_complete=datetime.datetime(2025, 2, 19, 22, 19, 33, 178906), params={'max_depth': 10, 'learning_rate': 0.0140717901612381, 'gamma': 2.18115950523157, 'min_child_weight': 9, 'subsample': 0.568153688660201, 'colsample_bytree': 0.838446676388734}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'learning_rate': FloatDistribution(high=0.3, log=False, low=0.01, step=None), 'gamma': FloatDistribution(high=5.0, log=False, low=0.0, step=None), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'subsample': FloatDistrib

In [ ]:
# Exibe os 10 melhores resultados
study_dmatrix.trials_dataframe().sort_values(by="value", ascending=True).head(10)

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_subsample,state
35,35,34.342077,2025-02-19 22:19:16.580193,2025-02-19 22:19:33.178906,0 days 00:00:16.598713,0.838447,2.181160,0.014072,10,9,0.568154,COMPLETE
34,34,34.413960,2025-02-19 22:19:00.254591,2025-02-19 22:19:16.579193,0 days 00:00:16.324602,0.837262,2.116301,0.013347,10,10,0.565440,COMPLETE
41,41,34.525381,2025-02-19 22:19:50.845453,2025-02-19 22:19:59.234331,0 days 00:00:08.388878,0.976480,3.378413,0.035674,10,10,0.560199,COMPLETE
48,48,34.526244,2025-02-19 22:20:28.954692,2025-02-19 22:20:35.701833,0 days 00:00:06.747141,0.941270,2.722686,0.039819,9,10,0.640604,COMPLETE
43,43,34.539823,2025-02-19 22:20:06.172980,2025-02-19 22:20:13.685824,0 days 00:00:07.512844,0.988524,3.654207,0.038029,9,10,0.618126,COMPLETE
40,40,34.558058,2025-02-19 22:19:43.727971,2025-02-19 22:19:50.844450,0 days 00:00:07.116479,0.960090,2.291526,0.036981,9,10,0.556583,COMPLETE
42,42,34.613824,2025-02-19 22:19:59.236330,2025-02-19 22:20:06.171981,0 days 00:00:06.935651,0.980267,3.460896,0.037993,9,10,0.605520,COMPLETE
31,31,34.616888,2025-02-19 22:18:21.328801,2025-02-19 22:18:37.357661,0 days 00:00:16.028860,0.824873,1.329688,0.012944,10,10,0.513995,COMPLETE
45,45,34.643637,2025-02-19 22:20:17.384811,2025-02-19 22:20:22.218916,0 days 00:00:04.834105,0.949136,3.486644,0.051161,9,2,0.558862,COMPLETE
33,33,34.658702,2025-02-19 22:18:43.504098,2025-02-19 22:19:00.253589,0 days 00:00:16.749491,0.816012,0.335119,0.010638,10,9,0.556253,COMPLETE


In [ ]:
study_dmatrix.best_params